# Google Auth - Google Authentication Library

Google Auth Library provides authentication mechanisms for Google APIs, including OAuth2, service accounts, and application default credentials.

## Example 1: Credentials and Authentication Basics

In [ ]:
# Example 1: AnonymousCredentials, the credential type for public resources
import google.auth
from google.auth import credentials, exceptions

print(f"google-auth version: {google.auth.__version__}")

anonymous = credentials.AnonymousCredentials()
assert isinstance(anonymous, credentials.Credentials)
print(f"valid={anonymous.valid} expired={anonymous.expired} token={anonymous.token}")
assert anonymous.valid and not anonymous.expired and anonymous.token is None

# Anonymous credentials never add an Authorization header to a request.
headers = {"x-demo": "1"}
anonymous.apply(headers)
anonymous.before_request(None, "GET", "https://example.com/resource", headers)
assert headers == {"x-demo": "1"}, headers

# There is nothing to refresh, and google-auth says so.
try:
    anonymous.refresh(None)
except exceptions.InvalidOperation as e:
    print(f"refresh rejected: {e}")
else:
    raise AssertionError("AnonymousCredentials.refresh() did not raise InvalidOperation")

## Example 2: OAuth2 and Token Management

In [ ]:
from google.oauth2 import credentials as oauth2_credentials
from google.auth import exceptions
import datetime

print("OAuth2 Token Management")

# Fake OAuth2 values: the credentials object is only built, never used against a server.
fake_token = "fake-access-token"
fake_refresh_token = "fake-refresh-token"
fake_client_id = "fake-client-id.apps.googleusercontent.com"
fake_client_secret = "fake-client-secret"

# Create OAuth2 credentials object (for demonstration)
try:
    oauth_creds = oauth2_credentials.Credentials(
        token=fake_token,
        refresh_token=fake_refresh_token,
        id_token=None,
        token_uri="https://oauth2.googleapis.com/token",
        client_id=fake_client_id,
        client_secret=fake_client_secret
    )
    
    print(f"OAuth2 Credentials created successfully")
    print(f"Token (first 10 chars): {oauth_creds.token[:10]}...")
    print(f"Has refresh token: {oauth_creds.refresh_token is not None}")
    print(f"Client ID: {oauth_creds.client_id}")
    print(f"Token URI: {oauth_creds.token_uri}")
    
    # Check if credentials are valid (without making network calls)
    print(f"\nCredential status:")
    print(f"Expired: {oauth_creds.expired}")
    print(f"Valid: {oauth_creds.valid}")
    
except Exception as e:
    print(f"Could not create OAuth2 credentials: {e}")
    raise

# A token with no expiry is never expired, so the credentials count as valid.
assert oauth_creds.token == fake_token and oauth_creds.refresh_token == fake_refresh_token
assert not oauth_creds.expired and oauth_creds.valid

# Demonstrate token scopes
scopes = [
    "https://www.googleapis.com/auth/cloud-platform",
    "https://www.googleapis.com/auth/bigquery",
    "https://www.googleapis.com/auth/storage"
]

print(f"\nCommon OAuth2 scopes:")
for scope in scopes:
    print(f"  - {scope}")

## Example 3: Signing and Verifying JWTs

In [ ]:
from google.auth import crypt, jwt
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import rsa
import base64
import json
import time

print("JWT signing and verification")

# A throwaway RSA key generated for this cell; it is never written anywhere.
key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
private_pem = key.private_bytes(
    serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8, serialization.NoEncryption()
)
public_pem = key.public_key().public_bytes(
    serialization.Encoding.PEM, serialization.PublicFormat.SubjectPublicKeyInfo
)
signer = crypt.RSASigner.from_string(private_pem, key_id="demo-key")

now = int(time.time())
payload = {
    "iss": "test@test-project.iam.gserviceaccount.com",
    "sub": "test@test-project.iam.gserviceaccount.com",
    "aud": "https://www.googleapis.com/oauth2/v4/token",
    "iat": now,
    "exp": now + 3600,
}
token = jwt.encode(signer, payload)
print(f"Signed token: {token[:16].decode()}... ({len(token)} bytes, 3 dot-separated parts)")
assert token.count(b".") == 2

# The first segment is the base64url-encoded header, which carries the algorithm and key id.
header_segment = token.split(b".")[0]
header = json.loads(base64.urlsafe_b64decode(header_segment + b"=" * (-len(header_segment) % 4)))
assert header["alg"] == "RS256" and header["kid"] == "demo-key", header
# Decoding without verification only parses the claims.
assert jwt.decode(token, verify=False) == payload

# Verified decode checks the signature, the audience and the expiry.
verified = jwt.decode(token, certs=public_pem, audience=payload["aud"])
assert verified["iss"] == payload["iss"] and verified["exp"] == payload["exp"]
print(f"Verified claims: iss={verified['iss']} aud={verified['aud']}")

# A token signed by a different key, or checked against another audience, is rejected.
other_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
other_pem = other_key.public_key().public_bytes(
    serialization.Encoding.PEM, serialization.PublicFormat.SubjectPublicKeyInfo
)
for certs, audience in ((other_pem, payload["aud"]), (public_pem, "https://other.example.com/")):
    try:
        jwt.decode(token, certs=certs, audience=audience)
    except ValueError as e:
        print(f"rejected: {e}")
    else:
        raise AssertionError("jwt.decode accepted a token it should have rejected")

## Example 4: Authentication Transport and Requests

In [ ]:
from google.auth.transport import requests as auth_requests
from google.auth.transport import urllib3 as auth_urllib3
import urllib3
import google.auth.transport._http_client as http_client

print("Authentication Transport Mechanisms")

# Test different transport options
transports = [
    ("requests", "HTTP transport using requests library"),
    ("urllib3", "HTTP transport using urllib3 library"),
    ("http_client", "Base HTTP client interface")
]

print("Available transport mechanisms:")
for name, description in transports:
    print(f"  {name}: {description}")

# Create request objects (without making actual requests)
print(f"\nTransport Configuration:")

try:
    # Create a requests-based transport
    request_transport = auth_requests.Request()
    print(f"✓ Requests transport created: {type(request_transport).__name__}")
except Exception as e:
    print(f"Could not create requests transport: {e}")
    raise

# Create urllib3-based transport: the pool manager is urllib3's own
# (google.auth.transport.urllib3 wraps one; it does not export PoolManager). Nothing is sent.
http = urllib3.PoolManager()
urllib3_transport = auth_urllib3.Request(http)
assert isinstance(urllib3_transport, auth_urllib3.Request) and urllib3_transport.http is http
print(f"✓ urllib3 transport created: {type(urllib3_transport).__name__}")

# Mock request configuration
mock_request_config = {
    "method": "POST",
    "url": "https://oauth2.googleapis.com/token",
    "headers": {
        "Content-Type": "application/x-www-form-urlencoded",
        "User-Agent": "google-auth-library-python/2.0.0"
    },
    "timeout": 30
}

print(f"\nExample token request configuration:")
for key, value in mock_request_config.items():
    print(f"  {key}: {value}")

## Example 5: Error Handling and Authentication Exceptions

In [ ]:
from google.auth import exceptions as auth_exceptions
from google.oauth2 import credentials

print("Google Auth Exception Handling")

# Test different authentication exception types
exception_types = [
    auth_exceptions.DefaultCredentialsError("Could not find default credentials"),
    auth_exceptions.RefreshError("Token refresh failed"),
    auth_exceptions.ReauthFailError("Reauthentication failed"),
    auth_exceptions.MalformedError("Invalid credential format"),
]

print("Authentication Exception Types:")
for exc in exception_types:
    print(f"  {exc.__class__.__name__}: {exc}")

# Test exception hierarchy
print(f"\nException Hierarchy Testing:")

# RefreshError must be caught as a GoogleAuthError
try:
    raise auth_exceptions.RefreshError("Mock refresh error")
except auth_exceptions.GoogleAuthError as e:
    assert type(e) is auth_exceptions.RefreshError, type(e)
    print(f"✓ RefreshError caught as GoogleAuthError: {e}")

try:
    raise auth_exceptions.DefaultCredentialsError("Mock credentials error")
except auth_exceptions.GoogleAuthError as e:
    print(f"✓ DefaultCredentialsError caught as GoogleAuthError: {e}")

# Test credential validation errors
print(f"\nCredential Validation:")

# Test with invalid token format
invalid_creds = credentials.Credentials(token="invalid_token_format")
assert invalid_creds.token == "invalid_token_format", invalid_creds.token
print("✓ Invalid token accepted (validation happens during use)")

print(f"\nAuthentication error handling tests completed!")